# 02 — Component Peaks & AEP Viewer

Identifies ERP component peak latencies from the 75 dB grand-average GFP (linked-ears
reference), saves them to `outputs/component_peaks.json`, then overlays the detected
components on GFP and per-channel AEP figures for both reference types.

**Workflow**
1. Run *Setup* cells (0–4) once to detect peaks and write the JSON.
2. Change `SUBJECT` in Settings and re-run from *Setup* onward as needed.
3. Run **View A** or **View B** independently to inspect figures with component overlays.


## 0. Imports


In [ ]:
%matplotlib widget
import sys, json
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import mne
from erp_tools.erp_viewer import ERPViewer

mne.set_log_level('WARNING')


## 1. Settings

Edit these before running the rest of the notebook.


In [ ]:
BIDS_ROOT = '/workspace/data/marmoset_FreqIntensity'
SUBJECTS  = ['Cj399', 'Cj459']   # all subjects — peak detection & JSON output
SUBJECT   = 'Cj399'              # single subject — viewer sections (Views A & B)

# Pipeline names for each reference type
PIPELINE_LE  = 'erp-pipeline-ref-linkedEars'
PIPELINE_CAR = 'erp-pipeline-ref-CAR'
PIPELINE_CMR = 'erp-pipeline-ref-CMR'

# Channel exclusions per reference (gfp_exclude, amp_exclude)
GFP_EXCLUDE_LE,  AMP_EXCLUDE_LE  = [],      ['A1', 'A2']
GFP_EXCLUDE_CAR, AMP_EXCLUDE_CAR = [],      []
GFP_EXCLUDE_CMR, AMP_EXCLUDE_CMR = [],      []

INTENSITIES = ['45dB', '60dB', '75dB']
FREQUENCIES = ['00125Hz', '00250Hz', '00500Hz', '01000Hz',
               '02000Hz', '04000Hz', '08000Hz', '16000Hz']

# Component search windows: (name, polarity, tmin_ms, tmax_ms)
COMPONENTS = [
    ('P1',  +1,  10,  20),
    ('N1',  -1,  30,  60),
    ('P2',  +1,  60, 100),
    ('N2',  -1, 120, 180),
    ('LN',  -1, 200, 600),
]

# Analysis window half-widths (ms) centred at detected peak
COMPONENT_WINDOW_MS = {'P1': 5/2, 'N1': 10/2, 'P2': 20/2, 'N2': 40/2, 'LN': 80/2}

# Colours used for component overlays (shared across GFP and AEP figures)
COMP_COLORS = {
    'P1': '#2196F3', 'N1': '#F44336', 'P2': '#4CAF50',
    'N2': '#FF9800', 'LN': '#9C27B0',
}

# ERPViewer display
GRID = [
    ['',   'Fz',  ''  ],
    ['C3', 'Cz',  'C4'],
    ['',   'Pz',  ''  ],
    ['A1', 'Oz', 'A2' ],
]
XLIM     = (-100, 650)   # ms
YLIM     = (-30, 20)   # µV  (set to None for auto-scale)
CI       = 0.95
AX_RATIO = 1.6

OUTPUT_DIR = Path('/workspace/outputs')
SAVE_DIR   = Path('/workspace/figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Helper functions


In [ ]:
def get_all_eeg_picks(evoked, gfp_exclude=[]):
    return mne.pick_types(evoked.info, eeg=True, exclude=gfp_exclude)


def compute_gfp(data_uv):
    """Population std (ddof=0) across channels. data_uv: (n_ch, n_times)."""
    return data_uv.std(axis=0, ddof=0)


def load_evokeds_for_subject(bids_root, subject, pipeline, intensities, frequencies):
    """Load condition-averaged evokeds. Returns {cond_label: mne.Evoked}."""
    deriv = Path(bids_root) / 'derivatives' / pipeline / f'sub-{subject}' / 'eeg'
    evokeds = {}
    for intensity in intensities:
        task = f'Passive{intensity}'
        for freq in frequencies:
            cond      = f'{intensity}{freq}'
            safe_cond = cond.replace('_', '').replace('-', '')
            fname     = deriv / f'sub-{subject}_task-{task}_cond-{safe_cond}_ave.fif'
            if fname.exists():
                try:
                    evokeds[cond] = mne.read_evokeds(fname, verbose=False)[0]
                except Exception as e:
                    print(f'  Warning: could not read {fname.name}: {e}')
    return evokeds


def draw_component_overlays(axes, subject, label=False):
    """Overlay component latency markers on one or more axes.

    axes  : matplotlib.Axes or iterable of Axes
    label : bool — add component-name label above each marker (use for GFP axis only)
    """
    if hasattr(axes, 'set_xlim'):
        axes = [axes]
    for comp_name, _, _, _ in COMPONENTS:
        lat = peak_latencies.get((subject, 'linkedEars', comp_name), np.nan)
        col = COMP_COLORS[comp_name]
        if np.isnan(lat):
            continue
        for ax in axes:
            ax.axvline(lat, color=col, lw=1.0, ls='-', alpha=0.8, zorder=3)
            if label:
                ax.text(lat, 1.0, comp_name,
                        transform=ax.get_xaxis_transform(),
                        ha='center', va='bottom', fontsize=7, color=col, clip_on=False)


## 3. Detect component peaks

Peak latencies are estimated from the 75 dB grand-average GFP using the linked-ears
reference. The same latencies are applied to both reference types in the viewer sections.


In [ ]:
LATENCY_INTENSITY = '75dB'

# peak_latencies[(subject, 'linkedEars', comp_name)] = latency_ms
peak_latencies = {}

print(f'Peak detection: linkedEars GFP, {LATENCY_INTENSITY}')
for _subj in SUBJECTS:
    _evokeds_75 = load_evokeds_for_subject(
        BIDS_ROOT, _subj, PIPELINE_LE, [LATENCY_INTENSITY], FREQUENCIES,
    )
    if not _evokeds_75:
        print(f'  sub-{_subj}: no 75 dB evoked files — skipping')
        continue

    _gfp_stack = []
    for _ev in _evokeds_75.values():
        _picks   = get_all_eeg_picks(_ev, GFP_EXCLUDE_LE)
        _data_uv = _ev.data[_picks] * 1e6
        _gfp_stack.append(compute_gfp(_data_uv))

    _mean_gfp = np.mean(_gfp_stack, axis=0)
    _times_ms = next(iter(_evokeds_75.values())).times * 1000

    print(f'  sub-{_subj} ({len(_gfp_stack)} frequencies):')
    for comp_name, _polarity, tmin_ms, tmax_ms in COMPONENTS:
        _mask = (_times_ms >= tmin_ms) & (_times_ms <= tmax_ms)
        if not _mask.any():
            lat_ms = np.nan
        else:
            _peak_idx = int(np.where(_mask)[0][np.argmax(_mean_gfp[_mask])])
            lat_ms    = float(_times_ms[_peak_idx])
        peak_latencies[(_subj, 'linkedEars', comp_name)] = lat_ms
        _disp = f'{lat_ms:.1f} ms' if not np.isnan(lat_ms) else 'not found'
        print(f'    {comp_name}: {_disp}')


### 3b. Analysis windows

Symmetric window of ±half-width centred at each detected peak.


In [ ]:
# analysis_windows[(subject, 'linkedEars', comp_name)] = (tmin_ms, tmax_ms)
analysis_windows = {}
for (_subj, _ref, comp_name), lat_ms in peak_latencies.items():
    half = COMPONENT_WINDOW_MS[comp_name]
    if np.isnan(lat_ms):
        analysis_windows[(_subj, _ref, comp_name)] = (np.nan, np.nan)
    else:
        analysis_windows[(_subj, _ref, comp_name)] = (lat_ms - half, lat_ms + half)

print(f'{"Subject":<10} {"Comp":<5}  {"Latency":>9}  {"Win tmin":>9}  {"Win tmax":>9}')
print('-' * 56)
for (_subj, _ref, comp_name), (w0, w1) in analysis_windows.items():
    lat = peak_latencies[(_subj, _ref, comp_name)]
    print(f'{_subj:<10} {comp_name:<5}  {lat:>8.1f}  {w0:>9.1f}  {w1:>9.1f}')


## 4. Save component info to JSON

Writes `outputs/component_peaks.json`. Notebook 03 reads this file to ensure
both notebooks use identical latency definitions.


In [ ]:
_comp_data = {
    'bids_root':         BIDS_ROOT,
    'latency_ref':       'linkedEars',
    'latency_intensity': LATENCY_INTENSITY,
    'subjects':          {},
}
for (_subj, _ref, comp_name), lat_ms in peak_latencies.items():
    w0, w1 = analysis_windows[(_subj, _ref, comp_name)]
    _comp_data['subjects'].setdefault(_subj, {})[comp_name] = {
        'latency_ms':     float(lat_ms) if not np.isnan(lat_ms) else None,
        'window_tmin_ms': float(w0)     if not np.isnan(w0)     else None,
        'window_tmax_ms': float(w1)     if not np.isnan(w1)     else None,
    }

_json_path = OUTPUT_DIR / 'component_peaks.json'
with open(_json_path, 'w') as _f:
    json.dump(_comp_data, _f, indent=2)
print(f'Saved: {_json_path}')
print(f'  Subjects: {list(_comp_data["subjects"])}')


## 5. Load epoch files

Loads clean epoch files for both reference types for the selected `SUBJECT`.
Re-run this cell (and Views below) after changing `SUBJECT` in Settings.


In [ ]:
def _load_epochs_all(pipeline, subject, intensities, bids_root=BIDS_ROOT):
    """Load clean epochs per intensity. Returns {intensity: mne.Epochs}."""
    deriv  = Path(bids_root) / 'derivatives' / pipeline / f'sub-{subject}' / 'eeg'
    epochs = {}
    for intensity in intensities:
        fif = deriv / f'sub-{subject}_task-Passive{intensity}_desc-clean_epo.fif'
        if not fif.exists():
            print(f'  Missing: {fif.name}')
            continue
        epo = mne.read_epochs(fif, preload=True, verbose=False)
        epochs[intensity] = epo
        print(f'  {intensity}: {len(epo)} epochs')
    return epochs

print(f'Loading linkedEars epochs — sub-{SUBJECT}')
epochs_le = _load_epochs_all(PIPELINE_LE, SUBJECT, INTENSITIES)

print(f'\nLoading CAR epochs — sub-{SUBJECT}')
epochs_car = _load_epochs_all(PIPELINE_CAR, SUBJECT, INTENSITIES)

print(f'\nLoading CMR epochs — sub-{SUBJECT}')
epochs_cmr = _load_epochs_all(PIPELINE_CMR, SUBJECT, INTENSITIES)


## 6. View A — Frequency tuning at fixed intensity

Four figures: **GFP** (linkedEars), **linkedEars AEP**, **CAR AEP**, **CMR AEP** — all with component overlays. Change `INTENSITY_TO_PLOT` and re-run the cell.


In [ ]:
# ── user settings ─────────────────────────────────────────────────────────────
INTENSITY_TO_PLOT = '75dB'   # '45dB', '60dB', or '75dB'
# FIGS_TO_SHOW = ['GFP', 'linkedEars', 'CAR', 'CMR']  # remove items to suppress any figure
FIGS_TO_SHOW = ['CMR']  # select from 'GFP', 'linkedEars', 'CAR', 'CMR'
SHOW_COMPONENT_OVERLAYS = True   # set False to hide component latency markers
LOG_SCALE_X = True              # set True to use log scale on the time axis

import ipywidgets as _w
from IPython.display import display as _display

_sel_freq_a = _w.SelectMultiple(
    options=FREQUENCIES, value=tuple(FREQUENCIES),
    description='Overlay:', rows=len(FREQUENCIES),
    layout=_w.Layout(width='210px'),
)
_btn_plot_a = _w.Button(description='Plot', button_style='success',
                         layout=_w.Layout(width='80px'))
_out_a  = _w.Output()
_figs_a = {}

def _apply_xscale(axes):
    """Apply log scale and positive-only xlim when LOG_SCALE_X is True."""
    if not LOG_SCALE_X:
        return
    for ax in axes:
        lo, hi = ax.get_xlim()
        ax.set_xscale('log')
        ax.set_xlim(max(lo, 1), hi)   # log scale requires positive lower bound

def _do_plot_a(_):
    for _f in _figs_a.values():
        if _f is not None:
            try: plt.close(_f)
            except Exception: pass
    _figs_a.clear()

    _sel = list(_sel_freq_a.value)
    if not _sel or INTENSITY_TO_PLOT not in epochs_le:
        with _out_a:
            _out_a.clear_output()
            print('No data — check INTENSITY_TO_PLOT and selection.')
        return

    _epo_le  = epochs_le[INTENSITY_TO_PLOT]
    _epo_car = epochs_car.get(INTENSITY_TO_PLOT)
    _epo_cmr = epochs_cmr.get(INTENSITY_TO_PLOT)

    _ce_le = {f'{INTENSITY_TO_PLOT}{f}': _epo_le[f'{INTENSITY_TO_PLOT}{f}']
              for f in _sel if f'{INTENSITY_TO_PLOT}{f}' in _epo_le.event_id}
    _ce_car = ({f'{INTENSITY_TO_PLOT}{f}': _epo_car[f'{INTENSITY_TO_PLOT}{f}']
                for f in _sel if f'{INTENSITY_TO_PLOT}{f}' in _epo_car.event_id}
               if _epo_car is not None else {})
    _ce_cmr = ({f'{INTENSITY_TO_PLOT}{f}': _epo_cmr[f'{INTENSITY_TO_PLOT}{f}']
                for f in _sel if f'{INTENSITY_TO_PLOT}{f}' in _epo_cmr.event_id}
               if _epo_cmr is not None else {})

    print(f'=== View A: {INTENSITY_TO_PLOT} — {len(_ce_le)} freq ===')

    v_le_a = ERPViewer(_ce_le, layout='grid', grid=GRID, ci=CI)

    # ── GFP ───────────────────────────────────────────────────────────────────
    _fig_gfp = None
    if 'GFP' in FIGS_TO_SHOW:
        _fig_gfp, _ax_gfp = plt.subplots(figsize=(9, 2.5))
        for _lbl, _epo in _ce_le.items():
            _ev  = _epo.average()
            _gfp = np.std(_ev.data, axis=0) * 1e6
            _ax_gfp.plot(_ev.times * 1000, _gfp, color=v_le_a.colors[_lbl], lw=0.9)
        _ax_gfp.axvline(0, color='k', lw=0.5, ls='--')
        _ax_gfp.set_xlim(XLIM)
        _ax_gfp.set_xlabel('Time (ms)')
        _ax_gfp.set_ylabel('GFP (µV)')
        _ax_gfp.set_title(f'sub-{SUBJECT}  {INTENSITY_TO_PLOT} — GFP by frequency', fontsize=10)
        if SHOW_COMPONENT_OVERLAYS:
            draw_component_overlays(_ax_gfp, SUBJECT, label=True)
        _apply_xscale([_ax_gfp])
        _fig_gfp.tight_layout()

    # ── linkedEars AEP ────────────────────────────────────────────────────────
    _fig_le = None
    if 'linkedEars' in FIGS_TO_SHOW:
        _fig_le = v_le_a.show(xlim=XLIM, ylim=YLIM if YLIM else None, ax_ratio=AX_RATIO)
        _fig_le.suptitle(f'sub-{SUBJECT}  {INTENSITY_TO_PLOT} — AEP linkedEars', fontsize=10)
        if SHOW_COMPONENT_OVERLAYS:
            draw_component_overlays(list(v_le_a.axes_map.values()), SUBJECT)
        _apply_xscale(list(v_le_a.axes_map.values()))

    # ── CAR AEP ───────────────────────────────────────────────────────────────
    _fig_car = None
    if 'CAR' in FIGS_TO_SHOW and _ce_car:
        _v_car = ERPViewer(_ce_car, layout='grid', grid=GRID, ci=CI, colors=v_le_a.colors)
        _fig_car = _v_car.show(xlim=XLIM, ylim=YLIM if YLIM else None, ax_ratio=AX_RATIO)
        _fig_car.suptitle(f'sub-{SUBJECT}  {INTENSITY_TO_PLOT} — AEP CAR', fontsize=10)
        if SHOW_COMPONENT_OVERLAYS:
            draw_component_overlays(list(_v_car.axes_map.values()), SUBJECT)
        _apply_xscale(list(_v_car.axes_map.values()))

    # ── CMR AEP ───────────────────────────────────────────────────────────────
    _fig_cmr = None
    if 'CMR' in FIGS_TO_SHOW and _ce_cmr:
        _v_cmr = ERPViewer(_ce_cmr, layout='grid', grid=GRID, ci=CI, colors=v_le_a.colors)
        _fig_cmr = _v_cmr.show(xlim=XLIM, ylim=YLIM if YLIM else None, ax_ratio=AX_RATIO)
        _fig_cmr.suptitle(f'sub-{SUBJECT}  {INTENSITY_TO_PLOT} — AEP CMR', fontsize=10)
        if SHOW_COMPONENT_OVERLAYS:
            draw_component_overlays(list(_v_cmr.axes_map.values()), SUBJECT)
        _apply_xscale(list(_v_cmr.axes_map.values()))

    _figs_a.update({'gfp': _fig_gfp, 'le': _fig_le, 'car': _fig_car, 'cmr': _fig_cmr})

    # ── save GUI ──────────────────────────────────────────────────────────────
    with _out_a:
        _out_a.clear_output(wait=True)
        _fmt    = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                              description='Format:', layout=_w.Layout(width='160px'))
        _sv_out = _w.Output()
        _rows   = []

        if _fig_gfp is not None:
            _n_gfp = _w.Text(value=f'{SUBJECT}_{INTENSITY_TO_PLOT}_gfp',
                             description='GFP:', layout=_w.Layout(width='320px'))
            _b_gfp = _w.Button(description='Save GFP', button_style='primary',
                               layout=_w.Layout(width='110px'))
            def _sv_gfp(_, f=_fig_gfp, n=_n_gfp, fmt=_fmt, o=_sv_out):
                with o:
                    o.clear_output()
                    path = SAVE_DIR / f'{n.value}.{fmt.value}'
                    f.savefig(str(path), format=fmt.value, bbox_inches='tight')
                    print(f'Saved: {path}')
            _b_gfp.on_click(_sv_gfp)
            _rows.append(_w.HBox([_n_gfp, _b_gfp]))

        if _fig_le is not None:
            _n_le = _w.Text(value=f'{SUBJECT}_{INTENSITY_TO_PLOT}_aep_linkedEars',
                            description='LE AEP:', layout=_w.Layout(width='320px'))
            _b_le = _w.Button(description='Save LE AEP', button_style='primary',
                              layout=_w.Layout(width='110px'))
            def _sv_le(_, f=_fig_le, n=_n_le, fmt=_fmt, o=_sv_out):
                with o:
                    o.clear_output()
                    path = SAVE_DIR / f'{n.value}.{fmt.value}'
                    f.savefig(str(path), format=fmt.value, bbox_inches='tight')
                    print(f'Saved: {path}')
            _b_le.on_click(_sv_le)
            _rows.append(_w.HBox([_n_le, _b_le]))

        if _fig_car is not None:
            _n_car = _w.Text(value=f'{SUBJECT}_{INTENSITY_TO_PLOT}_aep_CAR',
                             description='CAR AEP:', layout=_w.Layout(width='320px'))
            _b_car = _w.Button(description='Save CAR AEP', button_style='primary',
                               layout=_w.Layout(width='110px'))
            def _sv_car(_, f=_fig_car, n=_n_car, fmt=_fmt, o=_sv_out):
                with o:
                    o.clear_output()
                    path = SAVE_DIR / f'{n.value}.{fmt.value}'
                    f.savefig(str(path), format=fmt.value, bbox_inches='tight')
                    print(f'Saved: {path}')
            _b_car.on_click(_sv_car)
            _rows.append(_w.HBox([_n_car, _b_car]))

        if _fig_cmr is not None:
            _n_cmr = _w.Text(value=f'{SUBJECT}_{INTENSITY_TO_PLOT}_aep_CMR',
                             description='CMR AEP:', layout=_w.Layout(width='320px'))
            _b_cmr = _w.Button(description='Save CMR AEP', button_style='primary',
                               layout=_w.Layout(width='110px'))
            def _sv_cmr(_, f=_fig_cmr, n=_n_cmr, fmt=_fmt, o=_sv_out):
                with o:
                    o.clear_output()
                    path = SAVE_DIR / f'{n.value}.{fmt.value}'
                    f.savefig(str(path), format=fmt.value, bbox_inches='tight')
                    print(f'Saved: {path}')
            _b_cmr.on_click(_sv_cmr)
            _rows.append(_w.HBox([_n_cmr, _b_cmr]))

        if _rows:
            _display(*_rows, _fmt, _sv_out)

_btn_plot_a.on_click(_do_plot_a)
_display(_w.HBox([_sel_freq_a, _btn_plot_a]), _out_a)
_do_plot_a(None)


## 7. View B — Intensity comparison at fixed frequency

Four figures: **GFP** (linkedEars), **linkedEars AEP**, **CAR AEP**, **CMR AEP** — all with component overlays.
Change `FREQ_TO_PLOT` and re-run the cell.


In [ ]:
# ── user settings ─────────────────────────────────────────────────────────────
FREQ_TO_PLOT = '01000Hz'   # e.g. '00125Hz', '01000Hz', '08000Hz', '16000Hz'
FIGS_TO_SHOW = ['GFP', 'linkedEars', 'CAR', 'CMR']  # remove items to suppress any figure
SHOW_COMPONENT_OVERLAYS = True   # set False to hide component latency markers
LOG_SCALE_X = False              # set True to use log scale on the time axis

import ipywidgets as _w
from IPython.display import display as _display

_sel_int_b = _w.SelectMultiple(
    options=INTENSITIES, value=tuple(INTENSITIES),
    description='Overlay:', rows=len(INTENSITIES),
    layout=_w.Layout(width='210px'),
)
_btn_plot_b = _w.Button(description='Plot', button_style='success',
                         layout=_w.Layout(width='80px'))
_out_b  = _w.Output()
_figs_b = {}

def _apply_xscale_b(axes):
    """Apply log scale and positive-only xlim when LOG_SCALE_X is True."""
    if not LOG_SCALE_X:
        return
    for ax in axes:
        lo, hi = ax.get_xlim()
        ax.set_xscale('log')
        ax.set_xlim(max(lo, 1), hi)

def _do_plot_b(_):
    for _f in _figs_b.values():
        if _f is not None:
            try: plt.close(_f)
            except Exception: pass
    _figs_b.clear()

    _sel = list(_sel_int_b.value)
    if not _sel:
        with _out_b:
            _out_b.clear_output()
            print('Select at least one intensity.')
        return

    _ce_le  = {}
    _ce_car = {}
    _ce_cmr = {}
    for _intensity in _sel:
        _lbl = f'{_intensity}{FREQ_TO_PLOT}'
        if _intensity in epochs_le and _lbl in epochs_le[_intensity].event_id:
            _ce_le[_intensity] = epochs_le[_intensity][_lbl]
        if _intensity in epochs_car and _lbl in epochs_car[_intensity].event_id:
            _ce_car[_intensity] = epochs_car[_intensity][_lbl]
        if _intensity in epochs_cmr and _lbl in epochs_cmr[_intensity].event_id:
            _ce_cmr[_intensity] = epochs_cmr[_intensity][_lbl]

    if not _ce_le:
        with _out_b:
            _out_b.clear_output()
            print(f'{FREQ_TO_PLOT!r} not found — available: {FREQUENCIES}')
        return

    print(f'=== View B: {FREQ_TO_PLOT} — {len(_ce_le)} intensities ===')

    v_le_b = ERPViewer(_ce_le, layout='grid', grid=GRID, ci=CI)

    _hz     = int(FREQ_TO_PLOT.replace('Hz', ''))
    _hz_str = f'{_hz // 1000}k' if _hz >= 1000 else str(_hz)

    # ── GFP ───────────────────────────────────────────────────────────────────
    _fig_gfp = None
    if 'GFP' in FIGS_TO_SHOW:
        _fig_gfp, _ax_gfp = plt.subplots(figsize=(9, 2.5))
        for _intensity, _epo in _ce_le.items():
            _ev  = _epo.average()
            _gfp = np.std(_ev.data, axis=0) * 1e6
            _ax_gfp.plot(_ev.times * 1000, _gfp, color=v_le_b.colors[_intensity], lw=0.9)
        _ax_gfp.axvline(0, color='k', lw=0.5, ls='--')
        _ax_gfp.set_xlim(XLIM)
        _ax_gfp.set_xlabel('Time (ms)')
        _ax_gfp.set_ylabel('GFP (µV)')
        _ax_gfp.set_title(f'sub-{SUBJECT}  {FREQ_TO_PLOT} — GFP by intensity', fontsize=10)
        if SHOW_COMPONENT_OVERLAYS:
            draw_component_overlays(_ax_gfp, SUBJECT, label=True)
        _apply_xscale_b([_ax_gfp])
        _fig_gfp.tight_layout()

    # ── linkedEars AEP ────────────────────────────────────────────────────────
    _fig_le = None
    if 'linkedEars' in FIGS_TO_SHOW:
        _fig_le = v_le_b.show(xlim=XLIM, ylim=YLIM if YLIM else None, ax_ratio=AX_RATIO)
        _fig_le.suptitle(f'sub-{SUBJECT}  {FREQ_TO_PLOT} — AEP linkedEars', fontsize=10)
        if SHOW_COMPONENT_OVERLAYS:
            draw_component_overlays(list(v_le_b.axes_map.values()), SUBJECT)
        _apply_xscale_b(list(v_le_b.axes_map.values()))

    # ── CAR AEP ───────────────────────────────────────────────────────────────
    _fig_car = None
    if 'CAR' in FIGS_TO_SHOW and _ce_car:
        _v_car = ERPViewer(_ce_car, layout='grid', grid=GRID, ci=CI, colors=v_le_b.colors)
        _fig_car = _v_car.show(xlim=XLIM, ylim=YLIM if YLIM else None, ax_ratio=AX_RATIO)
        _fig_car.suptitle(f'sub-{SUBJECT}  {FREQ_TO_PLOT} — AEP car-ref', fontsize=10)
        if SHOW_COMPONENT_OVERLAYS:
            draw_component_overlays(list(_v_car.axes_map.values()), SUBJECT)
        _apply_xscale_b(list(_v_car.axes_map.values()))

    # ── CMR AEP ───────────────────────────────────────────────────────────────
    _fig_cmr = None
    if 'CMR' in FIGS_TO_SHOW and _ce_cmr:
        _v_cmr = ERPViewer(_ce_cmr, layout='grid', grid=GRID, ci=CI, colors=v_le_b.colors)
        _fig_cmr = _v_cmr.show(xlim=XLIM, ylim=YLIM if YLIM else None, ax_ratio=AX_RATIO)
        _fig_cmr.suptitle(f'sub-{SUBJECT}  {FREQ_TO_PLOT} — AEP CMR', fontsize=10)
        if SHOW_COMPONENT_OVERLAYS:
            draw_component_overlays(list(_v_cmr.axes_map.values()), SUBJECT)
        _apply_xscale_b(list(_v_cmr.axes_map.values()))

    _figs_b.update({'gfp': _fig_gfp, 'le': _fig_le,'car': _fig_car, 'cmr': _fig_cmr})

    # ── save GUI ──────────────────────────────────────────────────────────────
    with _out_b:
        _out_b.clear_output(wait=True)
        _fmt    = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                              description='Format:', layout=_w.Layout(width='160px'))
        _sv_out = _w.Output()
        _rows   = []

        if _fig_gfp is not None:
            _n_gfp = _w.Text(value=f'{SUBJECT}_{_hz_str}Hz_gfp',
                             description='GFP:', layout=_w.Layout(width='320px'))
            _b_gfp = _w.Button(description='Save GFP', button_style='primary',
                               layout=_w.Layout(width='110px'))
            def _sv_gfp(_, f=_fig_gfp, n=_n_gfp, fmt=_fmt, o=_sv_out):
                with o:
                    o.clear_output()
                    path = SAVE_DIR / f'{n.value}.{fmt.value}'
                    f.savefig(str(path), format=fmt.value, bbox_inches='tight')
                    print(f'Saved: {path}')
            _b_gfp.on_click(_sv_gfp)
            _rows.append(_w.HBox([_n_gfp, _b_gfp]))

        if _fig_le is not None:
            _n_le = _w.Text(value=f'{SUBJECT}_{_hz_str}Hz_aep_linkedEars',
                            description='LE AEP:', layout=_w.Layout(width='320px'))
            _b_le = _w.Button(description='Save LE AEP', button_style='primary',
                              layout=_w.Layout(width='110px'))
            def _sv_le(_, f=_fig_le, n=_n_le, fmt=_fmt, o=_sv_out):
                with o:
                    o.clear_output()
                    path = SAVE_DIR / f'{n.value}.{fmt.value}'
                    f.savefig(str(path), format=fmt.value, bbox_inches='tight')
                    print(f'Saved: {path}')
            _b_le.on_click(_sv_le)
            _rows.append(_w.HBox([_n_le, _b_le]))

        if _fig_car is not None:
            _n_car = _w.Text(value=f'{SUBJECT}_{_hz_str}Hz_aep_CAR',
                             description='CAR AEP:', layout=_w.Layout(width='320px'))
            _b_car = _w.Button(description='Save CAR AEP', button_style='primary',
                               layout=_w.Layout(width='110px'))
            def _sv_car(_, f=_fig_car, n=_n_car, fmt=_fmt, o=_sv_out):
                with o:
                    o.clear_output()
                    path = SAVE_DIR / f'{n.value}.{fmt.value}'
                    f.savefig(str(path), format=fmt.value, bbox_inches='tight')
                    print(f'Saved: {path}')
            _b_car.on_click(_sv_car)
            _rows.append(_w.HBox([_n_car, _b_car]))

        if _fig_cmr is not None:
            _n_cmr = _w.Text(value=f'{SUBJECT}_{_hz_str}Hz_aep_CMR',
                             description='CMR AEP:', layout=_w.Layout(width='320px'))
            _b_cmr = _w.Button(description='Save CMR AEP', button_style='primary',
                               layout=_w.Layout(width='110px'))
            def _sv_cmr(_, f=_fig_cmr, n=_n_cmr, fmt=_fmt, o=_sv_out):
                with o:
                    o.clear_output()
                    path = SAVE_DIR / f'{n.value}.{fmt.value}'
                    f.savefig(str(path), format=fmt.value, bbox_inches='tight')
                    print(f'Saved: {path}')
            _b_cmr.on_click(_sv_cmr)
            _rows.append(_w.HBox([_n_cmr, _b_cmr]))

        if _rows:
            _display(*_rows, _fmt, _sv_out)

_btn_plot_b.on_click(_do_plot_b)
_display(_w.HBox([_sel_int_b, _btn_plot_b]), _out_b)
_do_plot_b(None)
